In [31]:
import os
import sys
import subprocess
import ctypes
import sys as sys_lib

PROJECT_ROOT = '/Users/doanvinhnhan/Roo-Code'

if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 2. Import thư viện cốt lõi của Manim
from manim import *

# 3. Import toàn bộ thư viện custom của bạn

from skills.fami_lib import *
from skills.fami_assets_helper import *
from skills.fami_effects import * # Import thêm file này phòng trường hợp các hiệu ứng nằm ở đây
from skills.bit import BitSequence
from skills.broadcasting import Broadcasting
from skills.receiving import Receiving
config.tex_template = TexTemplate()
config.tex_template.add_to_preamble(r"\usepackage[utf8]{vietnam}")

TexTemplate(_body='', tex_compiler='latex', description='', output_format='.dvi', documentclass='\\documentclass[preview]{standalone}', preamble='\\usepackage[english]{babel}\n\\usepackage{amsmath}\n\\usepackage{amssymb}\n\\usepackage[utf8]{vietnam}', placeholder_text='YourTextHere', post_doc_commands='')

In [32]:
%%manim -v ERROR --resolution 720,1280 --frame_rate 15 --flush_cache --media_dir /Users/doanvinhnhan/Roo-Code/media BirthOfPixelReal

from manim import *
from PIL import Image
import numpy as np

class BirthOfPixelReal(Scene):
    def construct(self):
        # --------------------------------------------------
        # 1. TẢI ẢNH VÀ XỬ LÝ DỮ LIỆU THỰC TẾ
        # --------------------------------------------------
        img_path = "/Users/doanvinhnhan/Roo-Code/assets/Monalisa.jpg"
        
        try:
            pil_img = Image.open(img_path)
        except Exception as e:
            print(f"Lỗi tải ảnh: {e}")
            return

        img_w, img_h = pil_img.size
        aspect_ratio = img_w / img_h
        manim_h = 6.0
        manim_w = manim_h * aspect_ratio

        # Chuyển đổi sang Grayscale
        pil_gray = pil_img.convert('L')
        arr_gray = np.array(pil_gray) 
        
        color_mob = ImageMobject(pil_img)
        color_mob.height = manim_h
        
        gray_mob = ImageMobject(pil_gray)
        gray_mob.height = manim_h

        # --------------------------------------------------
        # 2. CÁC HÀM XỬ LÝ LƯỚI VÀ HIỆU ỨNG CHIA TÁCH
        # --------------------------------------------------
        
        # Hàm tạo lưới vector trả về mảng 2 chiều (để dễ dàng quản lý việc chia ô)
        def get_pixel_grid_2d(rows, cols):
            grid_2d = []
            cell_h_px = img_h / rows
            cell_w_px = img_w / cols
            rect_h = manim_h / rows
            rect_w = manim_w / cols
            
            for i in range(rows):
                row_list = []
                for j in range(cols):
                    # Trích xuất dữ liệu thực tế từ ma trận ảnh
                    r_start, r_end = int(i * cell_h_px), int((i + 1) * cell_h_px)
                    c_start, c_end = int(j * cell_w_px), int((j + 1) * cell_w_px)
                    
                    if r_start == r_end: r_end += 1
                    if c_start == c_end: c_end += 1
                    
                    block = arr_gray[r_start:r_end, c_start:c_end]
                    avg_val = int(np.mean(block))
                    
                    # Tạo ô vuông (không có số)
                    rect = Rectangle(width=rect_w, height=rect_h)
                    color_val = avg_val / 255.0
                    rect.set_fill(color=rgb_to_color([color_val, color_val, color_val]), opacity=1)
                    
                    # Giảm viền khi độ phân giải tăng
                    stroke_w = 1.5 if max(rows, cols) <= 16 else (0.5 if max(rows, cols) <= 32 else 0)
                    rect.set_stroke(GRAY, width=stroke_w)
                    
                    x = -manim_w/2 + (j + 0.5) * rect_w
                    y = manim_h/2 - (i + 0.5) * rect_h
                    rect.move_to(RIGHT * x + UP * y)
                    
                    row_list.append(rect)
                grid_2d.append(row_list)
            return grid_2d

        # Hàm chuyển mảng 2 chiều thành VGroup để Manim có thể hiển thị
        def flatten_2d(grid_2d):
            return VGroup(*[item for row in grid_2d for item in row])

        # HÀM QUAN TRỌNG: Tạo animation chia tách tại chỗ (1 ô tách thành 4 ô nhỏ)
        def subdivide_anim(old_2d, new_2d):
            anims = []
            old_rows, old_cols = len(old_2d), len(old_2d[0])
            
            for i in range(old_rows):
                for j in range(old_cols):
                    parent_cell = old_2d[i][j]
                    # Lấy 4 ô con tương ứng ở lưới mới (chia lưới 2x2 trong không gian của ô cha)
                    child1 = new_2d[2*i][2*j]
                    child2 = new_2d[2*i][2*j+1]
                    child3 = new_2d[2*i+1][2*j]
                    child4 = new_2d[2*i+1][2*j+1]
                    
                    children_group = VGroup(child1, child2, child3, child4)
                    # Transform 1 ô cha thành VGroup 4 ô con để triệt tiêu việc dịch chuyển
                    anims.append(ReplacementTransform(parent_cell, children_group))
            
            return AnimationGroup(*anims)

        # Hàm tạo ảnh pixel bằng thuật toán cho độ phân giải siêu cao (để tránh giật lag)
        def create_pixel_image(rows, cols):
            small = pil_gray.resize((cols, rows), Image.Resampling.BOX)
            pixelated = small.resize((img_w, img_h), Image.Resampling.NEAREST)
            mob = ImageMobject(pixelated)
            mob.height = manim_h
            return mob

        # --------------------------------------------------
        # 3. KỊCH BẢN ANIMATION
        # --------------------------------------------------
        
        # Mở đầu
        title = Text("Bước 1: Chuyển về thang xám (Grayscale)", font_size=36, color=YELLOW).to_edge(UP)
        self.play(FadeIn(color_mob), Write(title))
        self.wait(1)
        self.play(FadeIn(gray_mob))
        self.remove(color_mob)
        self.wait(1)
        
        # Bắt đầu với lưới 4x4
        grid_title = Text("Bước 2: Quá trình Lấy mẫu (Sampling)", font_size=36, color=YELLOW).to_edge(UP)
        
        grid_4x4 = get_pixel_grid_2d(4, 4)
        vgroup_4x4 = flatten_2d(grid_4x4)
        res_text = Text("Độ phân giải: 4 x 4", font_size=24).next_to(gray_mob, DOWN)
        
        self.play(
            Transform(title, grid_title),
            FadeOut(gray_mob),
            FadeIn(vgroup_4x4),
            Write(res_text)
        )
        self.wait(1)
        
        # 4x4 -> 8x8 (Tách lưới chuẩn xác)
        grid_8x8 = get_pixel_grid_2d(8, 8)
        text_8x8 = Text("Độ phân giải: 8 x 8", font_size=24).next_to(gray_mob, DOWN)
        self.play(subdivide_anim(grid_4x4, grid_8x8), ReplacementTransform(res_text, text_8x8), run_time=1.5)
        self.wait(0.5)
        
        # 8x8 -> 16x16
        grid_16x16 = get_pixel_grid_2d(16, 16)
        text_16x16 = Text("Độ phân giải: 16 x 16", font_size=24).next_to(gray_mob, DOWN)
        self.play(subdivide_anim(grid_8x8, grid_16x16), ReplacementTransform(text_8x8, text_16x16), run_time=1.5)
        self.wait(0.5)

        # 16x16 -> 32x32
        grid_32x32 = get_pixel_grid_2d(32, 32)
        text_32x32 = Text("Độ phân giải: 32 x 32", font_size=24).next_to(gray_mob, DOWN)
        self.play(subdivide_anim(grid_16x16, grid_32x32), ReplacementTransform(text_16x16, text_32x32), run_time=1.5)
        self.wait(0.5)
        
        # Chuyển tiếp từ Vector sang ImageMobject (Từ 64x64 trở đi để không bị lag máy)
        img_64x64 = create_pixel_image(64, 64)
        text_64x64 = Text("Độ phân giải: 64 x 64", font_size=24).next_to(gray_mob, DOWN)
        vgroup_32x32 = flatten_2d(grid_32x32)
        
        # Dùng FadeIn chồng lên vì pixel ở mức 64x64 đã đủ mịn
        self.play(
            FadeOut(vgroup_32x32),
            FadeIn(img_64x64),
            ReplacementTransform(text_32x32, text_64x64),
            run_time=1
        )
        current_img_mob = img_64x64
        res_text = text_64x64
        
        # Tăng dần độ phân giải siêu tốc lên đến 1/2 độ phân giải thực
        max_res = min(img_w // 2, img_h // 2)
        curr_val = 128
        
        while curr_val <= max_res:
            next_img_mob = create_pixel_image(curr_val, curr_val)
            next_res_text = Text(f"Độ phân giải: {curr_val} x {curr_val}", font_size=24).next_to(gray_mob, DOWN)
            
            self.play(
                FadeIn(next_img_mob),
                ReplacementTransform(res_text, next_res_text),
                run_time=0.6
            )
            self.remove(current_img_mob)
            current_img_mob = next_img_mob
            res_text = next_res_text
            curr_val *= 2
            
        # Nấc cuối cùng = 1/2 độ phân giải gốc
        if curr_val / 2 != max_res:
            final_img_mob = create_pixel_image(max_res, max_res)
            final_res_text = Text(f"Độ phân giải: {max_res} x {max_res} (1/2 Ảnh gốc)", font_size=24).next_to(gray_mob, DOWN)
            self.play(
                FadeIn(final_img_mob),
                ReplacementTransform(res_text, final_res_text),
                run_time=0.6
            )
            self.remove(current_img_mob)
            res_text = final_res_text
        
        self.wait(1)
        final_msg = Text("Không gian rời rạc càng dày, càng xấp xỉ liên tục!", font_size=32, color=GREEN_C).next_to(gray_mob, DOWN)
        self.play(ReplacementTransform(res_text, final_msg))
        self.wait(3)

Manim Community v0.19.0

KeyboardInterrupt: 

In [ ]:
%%manim -v ERROR --resolution 720,1280 --frame_rate 15 --flush_cache --media_dir /Users/doanvinhnhan/Roo-Code/media BirthOfPixelScript

from manim import *
from PIL import Image
import numpy as np

# Đảm bảo bạn đổi Base Class thành class hỗ trợ voiceover của bạn (ví dụ: VoiceoverScene)
class BirthOfPixelScript(FaMIBaseScene): 
    def construct(self):
        # --------------------------------------------------
        # 1. TẢI ẢNH VÀ XỬ LÝ
        # --------------------------------------------------
        img_path = "/Users/doanvinhnhan/Roo-Code/assets/Monalisa.jpg"
        
        try:
            pil_img = Image.open(img_path)
        except Exception as e:
            print(f"Lỗi tải ảnh: {e}")
            return

        img_w, img_h = pil_img.size
        aspect_ratio = img_w / img_h
        
        manim_h = 5.0 
        manim_w = manim_h * aspect_ratio
        image_shift = UP * 0.8

        pil_gray = pil_img.convert('L')
        arr_gray = np.array(pil_gray) 
        
        gray_mob = ImageMobject(pil_gray)
        gray_mob.height = manim_h
        gray_mob.shift(image_shift)

        # --------------------------------------------------
        # 2. CÁC HÀM XỬ LÝ LƯỚI VÀ CHIA TÁCH
        # --------------------------------------------------
        def get_pixel_grid_2d(rows, cols):
            grid_2d = []
            cell_h_px = img_h / rows
            cell_w_px = img_w / cols
            rect_h = manim_h / rows
            rect_w = manim_w / cols
            
            for i in range(rows):
                row_list = []
                for j in range(cols):
                    r_start, r_end = int(i * cell_h_px), int((i + 1) * cell_h_px)
                    c_start, c_end = int(j * cell_w_px), int((j + 1) * cell_w_px)
                    
                    if r_start == r_end: r_end += 1
                    if c_start == c_end: c_end += 1
                    
                    block = arr_gray[r_start:r_end, c_start:c_end]
                    avg_val = int(np.mean(block))
                    
                    rect = Rectangle(width=rect_w, height=rect_h)
                    color_val = avg_val / 255.0
                    rect.set_fill(color=rgb_to_color([color_val, color_val, color_val]), opacity=1)
                    
                    stroke_w = 1.5 if max(rows, cols) <= 16 else (0.5 if max(rows, cols) <= 32 else 0)
                    rect.set_stroke(GRAY, width=stroke_w)
                    
                    x = -manim_w/2 + (j + 0.5) * rect_w
                    y = manim_h/2 - (i + 0.5) * rect_h
                    rect.move_to(RIGHT * x + UP * y + image_shift)
                    
                    row_list.append(rect)
                grid_2d.append(row_list)
            return grid_2d

        def flatten_2d(grid_2d):
            return VGroup(*[item for row in grid_2d for item in row])

        def subdivide_anim(old_2d, new_2d):
            anims = []
            old_rows, old_cols = len(old_2d), len(old_2d[0])
            for i in range(old_rows):
                for j in range(old_cols):
                    parent_cell = old_2d[i][j]
                    child1 = new_2d[2*i][2*j]
                    child2 = new_2d[2*i][2*j+1]
                    child3 = new_2d[2*i+1][2*j]
                    child4 = new_2d[2*i+1][2*j+1]
                    children_group = VGroup(child1, child2, child3, child4)
                    anims.append(ReplacementTransform(parent_cell, children_group))
            return AnimationGroup(*anims)

        def create_pixel_image(rows, cols):
            small = pil_gray.resize((cols, rows), Image.Resampling.BOX)
            pixelated = small.resize((img_w, img_h), Image.Resampling.NEAREST)
            mob = ImageMobject(pixelated)
            mob.height = manim_h
            mob.shift(image_shift) 
            return mob

        # --------------------------------------------------
        # 3. KỊCH BẢN ANIMATION KẾT HỢP VOICEOVER & SUB
        # --------------------------------------------------
        
        # [HOOK & GIỚI THIỆU]
        title = Text("Bức ảnh trong máy tính thực chất là gì?", font_size=32, color=YELLOW).next_to(gray_mob, DOWN, buff=0.5)
        
        with self.voiceover(text="Có lẽ bạn đã biết các bức ảnh kỹ thuật số, thực ra là các pixel, nhưng bạn có thực sự hiểu các pixel là gì?") as tracker:
            self.update_subtitle("Có lẽ bạn đã biết các bức ảnh kỹ thuật số,\nthực ra là các pixel, nhưng bạn có thực sự hiểu pixel là gì?")
            self.play(FadeIn(gray_mob), Write(title))

        with self.voiceover(text="Tại sao lại có thể chuyển từ hình ảnh từ thế giới thực trở thành các bức ảnh nằm trong máy tính? Các file JPG, PNG thực ra là gì?") as tracker:
            self.update_subtitle("Tại sao có thể đưa hình ảnh từ thế giới thực vào máy tính?\nCác file JPG, PNG thực ra là gì?")
            # Giữ nguyên khung hình cho khán giả thẩm thấu

        with self.voiceover(text="Trong video này, tôi sẽ giải thích cho các bạn về các khái niệm này! Đầu tiên, ta cùng tìm hiểu định nghĩa cơ bản nhất của một hình ảnh kỹ thuật số.") as tracker:
            self.update_subtitle("Trong video này, tôi sẽ giải thích về các khái niệm này!\nĐầu tiên, hãy tìm hiểu định nghĩa cơ bản của hình ảnh kỹ thuật số.")
            self.play(FadeOut(title))
        
        # [ĐỊNH NGHĨA TOÁN HỌC]
        func_text = MathTex(r"f(x, y)", font_size=48)
        desc_text = Text("Tọa độ không gian liên tục", font_size=24)
        func_group = VGroup(func_text, desc_text).arrange(DOWN, buff=0.2).next_to(gray_mob, DOWN, buff=0.4)
        
        with self.voiceover(text="Một hình ảnh ở không gian vật lý có thể được định nghĩa là một hàm số f(x,y) trong đó x và y là các tọa độ không gian liên tục trong thế giới thực, giá trị của hàm số tại x,y được gọi là cường độ ánh sáng hay mức xám.") as tracker:
            self.update_subtitle("Hình ảnh vật lý được định nghĩa là hàm số f(x,y),\nvới x, y là các tọa độ không gian liên tục.")
            self.play(FadeIn(func_group))

        discrete_text = Text("Xấp xỉ bằng tập hợp hữu hạn rời rạc", font_size=24, color=RED_B).move_to(desc_text.get_center())
        
        with self.voiceover(text="Và ta đều biết, máy tính chỉ có thể lưu trữ các tập hợp hữu hạn. Do nguyên nhân đó, ta xấp xỉ không gian x,y và hàm số f(x,y) bằng các đại lượng rời rạc, cách đều nhau và hữu hạn.") as tracker:
            self.update_subtitle("Máy tính chỉ có thể lưu trữ tập hợp hữu hạn.\nDo đó ta xấp xỉ không gian bằng các đại lượng rời rạc, cách đều nhau.")
            self.play(ReplacementTransform(desc_text, discrete_text))

        with self.voiceover(text="Hàm số không gian liên tục - Hình ảnh vật lý này từ đó chuyển đổi thành Hàm số rời rạc - một hình ảnh kỹ thuật số.") as tracker:
            self.update_subtitle("Hàm số không gian liên tục từ đó chuyển đổi thành\nHàm số rời rạc - một hình ảnh kỹ thuật số.")
            # Chờ sync giọng đọc

        with self.voiceover(text="Và magic bắt đầu ở đây, quá trình chuyển đổi từ liên tục sang rời rạc này được thực hiện thông qua hai bước cốt lõi:") as tracker:
            self.update_subtitle("Và magic bắt đầu ở đây, quá trình chuyển đổi từ 'liên tục'\nsang 'rời rạc' được thực hiện qua hai bước cốt lõi:")

        # [BƯỚC 1: LẤY MẪU]
        step1_title = Text("1. Lấy mẫu - Sự ra đời của Pixel", font_size=32, color=YELLOW).next_to(gray_mob, DOWN, buff=0.5)
        grid_lines = NumberPlane(
            x_range=[-manim_w/2, manim_w/2, manim_w/8],
            y_range=[-manim_h/2, manim_h/2, manim_h/8],
            background_line_style={"stroke_color": WHITE, "stroke_width": 2, "stroke_opacity": 0.8}
        ).move_to(gray_mob.get_center())

        with self.voiceover(text="1. Lấy mẫu - Sự ra đời của Pixel: Hãy tưởng tượng bạn đặt một tấm lưới lên trên bức ảnh vật lý thực tế.") as tracker:
            self.update_subtitle("1. Lấy mẫu: Tưởng tượng bạn đặt một tấm lưới\nlên trên bức ảnh vật lý thực tế.")
            self.play(FadeOut(func_text), FadeOut(discrete_text), Write(step1_title))
            self.play(Create(grid_lines), run_time=2)

        with self.voiceover(text="Mỗi ô vuông nhỏ trên tấm lưới đó sẽ đại diện cho một tọa độ không gian rời rạc. Ta gọi mỗi ô vuông này là một Pixel, viết tắt của Picture Element.") as tracker:
            self.update_subtitle("Mỗi ô vuông nhỏ đại diện cho một tọa độ không gian rời rạc.\nTa gọi mỗi ô vuông này là một Pixel.")

        # [BƯỚC 2: LƯỢNG TỬ HÓA]
        step2_title = Text("2. Lượng tử hóa", font_size=32, color=YELLOW)
        matrix_desc = Text("Ma trận 2 chiều chứa số nguyên (0 - 255)", font_size=24, color=GREEN_C)
        step2_group = VGroup(step2_title, matrix_desc).arrange(DOWN, buff=0.2).next_to(gray_mob, DOWN, buff=0.4)
        
        grid_8x8 = get_pixel_grid_2d(8, 8)
        vgroup_8x8 = flatten_2d(grid_8x8)
        
        numbers_group = VGroup()
        for row in grid_8x8:
            for rect in row:
                color_rgb = color_to_rgb(rect.get_fill_color())
                val = int(color_rgb[0] * 255)
                num_text = Text(str(val), font_size=16)
                num_text.set_color(BLACK if val > 127 else WHITE)
                num_text.move_to(rect.get_center())
                numbers_group.add(num_text)
                
        left_bracket = Text("[", font_size=180, weight=BOLD).next_to(vgroup_8x8, LEFT, buff=0.1)
        right_bracket = Text("]", font_size=180, weight=BOLD).next_to(vgroup_8x8, RIGHT, buff=0.1)

        with self.voiceover(text="2. Lượng tử hóa: Ta thực hiện gán một số nguyên đại diện cho cường độ tại pixel đó, thông thường nằm trong khoảng 0 đến 255.") as tracker:
            self.update_subtitle("2. Lượng tử hóa: Ta gán một số nguyên đại diện\ncho cường độ tại pixel đó (0 - 255).")
            self.play(ReplacementTransform(step1_title, step2_group[0]))
            self.play(FadeOut(gray_mob), FadeOut(grid_lines), FadeIn(vgroup_8x8))
            self.play(FadeIn(numbers_group))

        with self.voiceover(text="Vậy là một bức ảnh đen trắng trong máy tính, thực chất chỉ là một ma trận 2 chiều chứa các con số 0 đến 255.") as tracker:
            self.update_subtitle("Vậy là một bức ảnh trong máy tính thực chất chỉ là\nmột ma trận 2 chiều chứa các con số 0 đến 255.")
            self.play(Write(left_bracket), Write(right_bracket), FadeIn(step2_group[1]))

        # [ĐỘ PHÂN GIẢI]
        res_title = Text("Độ phân giải (Resolution)", font_size=32, color=YELLOW)
        res_desc = Text("Lưới càng nhỏ, pixel càng nhiều -> Xấp xỉ càng sát thực tế", font_size=24, color=WHITE)
        res_group = VGroup(res_title, res_desc).arrange(DOWN, buff=0.2).next_to(vgroup_8x8, DOWN, buff=0.4)
        
        grid_16x16 = get_pixel_grid_2d(16, 16)
        grid_32x32 = get_pixel_grid_2d(32, 32)

        with self.voiceover(text="Và nếu bạn tăng số ô chia lưới lên, lưới càng được chia nhỏ, số lượng pixel càng nhiều, thì bức ảnh kỹ thuật số càng chi tiết và xấp xỉ càng sát với hình ảnh trong thế giới thực.") as tracker:
            self.update_subtitle("Nếu tăng số ô chia lưới, số lượng pixel càng nhiều,\nthì bức ảnh càng chi tiết và sát với thực tế.")
            self.play(
                ReplacementTransform(step2_group, res_group),
                FadeOut(numbers_group), FadeOut(left_bracket), FadeOut(right_bracket)
            )
            # Chạy animation chia nhỏ trong lúc đọc
            self.play(subdivide_anim(grid_8x8, grid_16x16), run_time=1.5)
            self.play(subdivide_anim(grid_16x16, grid_32x32), run_time=1.5)

        with self.voiceover(text="Đó chính là lý do ta gọi số lượng ô chiều ngang và chiều dọc là độ phân giải.") as tracker:
            self.update_subtitle("Đó chính là lý do ta gọi số lượng ô chiều ngang\nvà chiều dọc là Độ phân giải.")
            
            # Đẩy nhanh tới ảnh độ nét cao trong câu chốt
            img_64x64 = create_pixel_image(64, 64)
            vgroup_32x32 = flatten_2d(grid_32x32)
            self.play(FadeOut(vgroup_32x32), FadeIn(img_64x64), run_time=0.5)
            
            current_img_mob = img_64x64
            max_res = min(img_w // 2, img_h // 2)
            curr_val = 128
            
            while curr_val <= max_res:
                next_img_mob = create_pixel_image(curr_val, curr_val)
                self.play(FadeIn(next_img_mob), run_time=0.3)
                self.remove(current_img_mob)
                current_img_mob = next_img_mob
                curr_val *= 2
                
            if curr_val / 2 != max_res:
                final_img_mob = create_pixel_image(max_res, max_res)
                self.play(FadeIn(final_img_mob), run_time=0.3)
                self.remove(current_img_mob)

Manim Community v0.19.0

[05/09/26 00:49:19] WARNING  Some options were not used: {'shortest': '1', 'metadata':     ]8;id=122032;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=761790;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#801\801]8;;\
                             'comment=Rendered with Manim Community v0.19.0'}                                      

In [ ]:
%%manim -v ERROR --resolution 720,1280 --frame_rate 15 --flush_cache --media_dir /Users/doanvinhnhan/Roo-Code/media BirthOfPixelScript

import os
import sys
sys_lib = sys

import numpy as np
from PIL import Image

PROJECT_ROOT = '/Users/doanvinhnhan/Roo-Code'

if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from manim import *

# Cố gắng import FaMIBaseScene từ thư viện của bạn
try:
    from skills.fami_lib import FaMIBaseScene
except ImportError:
    from manim_voiceover import VoiceoverScene
    FaMIBaseScene = VoiceoverScene

class BirthOfPixelScript(FaMIBaseScene): 
    def construct(self):
        # --------------------------------------------------
        # 1. TẢI ẢNH VÀ XỬ LÝ KÍCH THƯỚC
        # --------------------------------------------------
        img_path = "/Users/doanvinhnhan/Roo-Code/assets/Monalisa.jpg"
        
        try:
            pil_img = Image.open(img_path)
        except Exception as e:
            print(f"Lỗi tải ảnh: {e}")
            return

        img_w, img_h = pil_img.size
        aspect_ratio = img_w / img_h
        
        manim_h = 5.0 
        manim_w = manim_h * aspect_ratio
        image_shift = UP * 0.8

        pil_gray = pil_img.convert('L')
        arr_gray = np.array(pil_gray) 
        
        gray_mob = ImageMobject(pil_gray)
        gray_mob.height = manim_h
        gray_mob.shift(image_shift)

        # --------------------------------------------------
        # 2. CÁC HÀM XỬ LÝ LƯỚI VÀ CHIA TÁCH
        # --------------------------------------------------
        def get_pixel_grid_2d(rows, cols):
            grid_2d = []
            cell_h_px = img_h / rows
            cell_w_px = img_w / cols
            rect_h = manim_h / rows
            rect_w = manim_w / cols
            
            for i in range(rows):
                row_list = []
                for j in range(cols):
                    r_start, r_end = int(i * cell_h_px), int((i + 1) * cell_h_px)
                    c_start, c_end = int(j * cell_w_px), int((j + 1) * cell_w_px)
                    if r_start == r_end: r_end += 1
                    if c_start == c_end: c_end += 1
                    
                    block = arr_gray[r_start:r_end, c_start:c_end]
                    avg_val = int(np.mean(block))
                    
                    rect = Rectangle(width=rect_w, height=rect_h)
                    color_val = avg_val / 255.0
                    rect.set_fill(color=rgb_to_color([color_val, color_val, color_val]), opacity=1)
                    
                    stroke_w = 1.5 if max(rows, cols) <= 16 else (0.5 if max(rows, cols) <= 32 else 0)
                    rect.set_stroke(GRAY, width=stroke_w)
                    
                    x = -manim_w/2 + (j + 0.5) * rect_w
                    y = manim_h/2 - (i + 0.5) * rect_h
                    rect.move_to(RIGHT * x + UP * y + image_shift)
                    row_list.append(rect)
                grid_2d.append(row_list)
            return grid_2d

        def flatten_2d(grid_2d):
            return VGroup(*[item for row in grid_2d for item in row])

        def subdivide_anim(old_2d, new_2d):
            anims = []
            old_rows, old_cols = len(old_2d), len(old_2d[0])
            for i in range(old_rows):
                for j in range(old_cols):
                    parent_cell = old_2d[i][j]
                    children_group = VGroup(
                        new_2d[2*i][2*j], new_2d[2*i][2*j+1],
                        new_2d[2*i+1][2*j], new_2d[2*i+1][2*j+1]
                    )
                    anims.append(ReplacementTransform(parent_cell, children_group))
            return AnimationGroup(*anims)

        def create_pixel_image(rows, cols):
            small = pil_gray.resize((cols, rows), Image.Resampling.BOX)
            pixelated = small.resize((img_w, img_h), Image.Resampling.NEAREST)
            mob = ImageMobject(pixelated)
            mob.height = manim_h
            mob.shift(image_shift) 
            return mob

        # --------------------------------------------------
        # 3. KỊCH BẢN KẾT HỢP VOICEOVER & SUB
        # --------------------------------------------------
        
        title = self.create_title("Bức ảnh trong máy tính thực chất là gì?")
        
        # Tạo khung ảnh ảo (.PNG & .JPG)
        generic_frame = Rectangle(width=5, height=4, color=WHITE).shift(image_shift)
        mountain = Polygon(
            generic_frame.get_corner(DL),
            generic_frame.get_bottom() + UP*2,
            generic_frame.get_corner(DR),
            color=GRAY, fill_opacity=0.3
        )
        sun = Circle(radius=0.5, color=YELLOW, fill_opacity=0.8).move_to(generic_frame.get_corner(UR) + DL*1.2)
        generic_pic = VGroup(generic_frame, mountain, sun)
        
        png_text = Text(".PNG", font_size=40, color=BLUE).move_to(generic_frame.get_center() + LEFT*1.2)
        jpg_text = Text(".JPG", font_size=40, color=ORANGE).move_to(generic_frame.get_center() + RIGHT*1.2)

        # [ĐOẠN 1: HOOK]
        with self.voiceover(text="Có lẽ bạn đã biết các bức ảnh kỹ thuật số, thực ra là các pixel, nhưng bạn có thực sự hiểu các pixel là gì?") as tracker:
            self.update_subtitle("Có lẽ bạn đã biết các bức ảnh kỹ thuật số,\nthực ra là các pixel, nhưng bạn có thực sự hiểu pixel là gì?")
            self.play(Write(title), Create(generic_pic), Write(png_text), Write(jpg_text))

        with self.voiceover(text="Tại sao lại có thể chuyển từ hình ảnh từ thế giới thực trở thành các bức ảnh nằm trong máy tính? Các file JPG, PNG thực ra là gì?") as tracker:
            self.update_subtitle("Tại sao có thể đưa hình ảnh từ thế giới thực vào máy tính?\nCác file JPG, PNG thực ra là gì?")
            self.play(Indicate(png_text), Indicate(jpg_text))

        with self.voiceover(text="Trong video này, tôi sẽ giải thích cho các bạn về các khái niệm này! Đầu tiên, ta cùng tìm hiểu định nghĩa cơ bản nhất của một hình ảnh kỹ thuật số.") as tracker:
            self.update_subtitle("Trong video này, tôi sẽ giải thích về các khái niệm này!\nĐầu tiên, hãy tìm hiểu định nghĩa cơ bản của hình ảnh kỹ thuật số.")
            self.play(FadeOut(generic_pic, png_text, jpg_text, title))
        
        # [ĐOẠN 2: ĐỊNH NGHĨA VẬT LÝ VÀ HÀM SỐ]
        func_text = Text("f(x, y)", font_size=40, slant=ITALIC).next_to(gray_mob, RIGHT, buff=0.5)
        func_arrow = Arrow(gray_mob.get_right(), func_text.get_left(), buff=0.2, color=YELLOW)
        
        # Tạo trục x, y bám sát viền ảnh
        axes_origin = gray_mob.get_corner(DL)
        x_axis = Line(axes_origin, axes_origin + RIGHT*(manim_w + 0.5), color=BLUE).add_tip()
        y_axis = Line(axes_origin, axes_origin + UP*(manim_h + 0.5), color=RED).add_tip()
        
        x_label = Text("x", font_size=32, slant=ITALIC).next_to(x_axis.get_end(), DOWN)
        y_label = Text("y", font_size=32, slant=ITALIC).next_to(y_axis.get_end(), LEFT)
        axes_group = VGroup(x_axis, y_axis, x_label, y_label)

        with self.voiceover(text="Một hình ảnh ở không gian vật lý có thể được định nghĩa là một hàm số f(x,y) trong đó x và y là các tọa độ không gian liên tục trong thế giới thực, giá trị của hàm số tại x,y được gọi là cường độ ánh sáng hay mức xám.") as tracker:
            self.update_subtitle("Hình ảnh vật lý được định nghĩa là hàm số f(x,y),\nvới x, y là các tọa độ không gian liên tục.")
            
            self.play(FadeIn(gray_mob))
            self.play(Create(axes_group), run_time=1)
            self.play(Write(func_text), Create(func_arrow))
            
            # Lưu lại trạng thái ảnh gốc để khôi phục sau khi Zoom
            gray_mob.save_state()
            self.bring_to_front(gray_mob) 
                        
        # [ĐOẠN 3: XẤP XỈ RỜI RẠC]
        discrete_text = Text("Xấp xỉ bằng tập hợp hữu hạn rời rạc", font_size=24, color=RED_B).next_to(gray_mob, DOWN, buff=0.4)
        
        with self.voiceover(text="Và ta đều biết, máy tính chỉ có thể lưu trữ các tập hợp hữu hạn. Do nguyên nhân đó, ta xấp xỉ không gian x,y và hàm số f(x,y) bằng các đại lượng rời rạc, cách đều nhau và hữu hạn.") as tracker:
            self.update_subtitle("Máy tính chỉ lưu trữ tập hợp hữu hạn.\nDo đó ta xấp xỉ hàm số bằng đại lượng rời rạc, hữu hạn.")
            
            self.play(Restore(gray_mob), run_time=1.5) 
            self.play(FadeOut(axes_group, func_arrow, func_text)) 
            self.play(Write(discrete_text))

        with self.voiceover(text="Hàm số không gian liên tục - Hình ảnh vật lý này từ đó chuyển đổi thành Hàm số rời rạc - một hình ảnh kỹ thuật số.") as tracker:
            self.update_subtitle("Hàm số không gian liên tục từ đó chuyển đổi thành\nHàm số rời rạc - một hình ảnh kỹ thuật số.")

        with self.voiceover(text="Và magic bắt đầu ở đây, quá trình chuyển đổi từ liên tục sang rời rạc này được thực hiện thông qua hai bước cốt lõi:") as tracker:
            self.update_subtitle("Magic bắt đầu ở đây, quá trình chuyển đổi từ 'liên tục'\nsang 'rời rạc' được thực hiện qua hai bước cốt lõi:")
            self.play(FadeOut(discrete_text))

        # [ĐOẠN 4: LẤY MẪU & LƯỢNG TỬ HÓA]
        step1_title = Text("1. Lấy mẫu - Sự ra đời của Pixel", font_size=32, color=YELLOW).next_to(gray_mob, DOWN, buff=0.5)
        grid_lines = NumberPlane(
            x_range=[-manim_w/2, manim_w/2, manim_w/8], y_range=[-manim_h/2, manim_h/2, manim_h/8],
            background_line_style={"stroke_color": WHITE, "stroke_width": 2, "stroke_opacity": 0.8}
        ).move_to(gray_mob.get_center())

        with self.voiceover(text="1. Lấy mẫu - Sự ra đời của Pixel: Hãy tưởng tượng bạn đặt một tấm lưới lên trên bức ảnh vật lý thực tế.") as tracker:
            self.update_subtitle("1. Lấy mẫu: Tưởng tượng bạn đặt một tấm lưới\nlên trên bức ảnh vật lý thực tế.")
            self.play(Write(step1_title))
            self.play(Create(grid_lines), run_time=2)

        with self.voiceover(text="Mỗi ô vuông nhỏ trên tấm lưới đó sẽ đại diện cho một tọa độ không gian rời rạc. Ta gọi mỗi ô vuông này là một Pixel, viết tắt của Picture Element.") as tracker:
            self.update_subtitle("Mỗi ô vuông nhỏ đại diện cho một tọa độ không gian rời rạc.\nTa gọi mỗi ô vuông này là một Pixel.")

        step2_title = Text("2. Lượng tử hóa", font_size=32, color=YELLOW)
        matrix_desc = Text("Ma trận 2 chiều chứa số nguyên (0 - 255)", font_size=24, color=GREEN_C)
        step2_group = VGroup(step2_title, matrix_desc).arrange(DOWN, buff=0.2).next_to(gray_mob, DOWN, buff=0.4)
        
        grid_8x8 = get_pixel_grid_2d(8, 8)
        vgroup_8x8 = flatten_2d(grid_8x8)
        
        numbers_group = VGroup()
        for row in grid_8x8:
            for rect in row:
                color_rgb = color_to_rgb(rect.get_fill_color())
                val = int(color_rgb[0] * 255)
                num_text = Text(str(val), font_size=16)
                num_text.set_color(BLACK if val > 127 else WHITE)
                num_text.move_to(rect.get_center())
                numbers_group.add(num_text)
                
        left_bracket = Text("[", font_size=180, weight=BOLD).next_to(vgroup_8x8, LEFT, buff=0.1)
        right_bracket = Text("]", font_size=180, weight=BOLD).next_to(vgroup_8x8, RIGHT, buff=0.1)

        with self.voiceover(text="2. Lượng tử hóa: Ta thực hiện gán một số nguyên đại diện cho cường độ tại pixel đó, thông thường nằm trong khoảng 0 đến 255.") as tracker:
            self.update_subtitle("2. Lượng tử hóa: Ta gán một số nguyên đại diện\ncho cường độ tại pixel đó (0 - 255).")
            self.play(ReplacementTransform(step1_title, step2_group[0]))
            self.play(FadeOut(gray_mob), FadeOut(grid_lines), FadeIn(vgroup_8x8))
            self.play(FadeIn(numbers_group))

        with self.voiceover(text="Vậy là một bức ảnh đen trắng trong máy tính, thực chất chỉ là một ma trận 2 chiều chứa các con số 0 đến 255.") as tracker:
            self.update_subtitle("Vậy là một bức ảnh trong máy tính thực chất chỉ là\nmột ma trận 2 chiều chứa các con số 0 đến 255.")
            self.play(Write(left_bracket), Write(right_bracket), FadeIn(step2_group[1]))

        # [ĐOẠN 5: ĐỘ PHÂN GIẢI]
        res_title = Text("Độ phân giải (Resolution)", font_size=32, color=YELLOW)
        res_desc = Text("Lưới càng nhỏ, pixel càng nhiều -> Xấp xỉ càng sát thực tế", font_size=24, color=WHITE)
        res_group = VGroup(res_title, res_desc).arrange(DOWN, buff=0.2).next_to(vgroup_8x8, DOWN, buff=0.4)
        
        grid_16x16 = get_pixel_grid_2d(16, 16)
        grid_32x32 = get_pixel_grid_2d(32, 32)

        with self.voiceover(text="Và nếu bạn tăng số ô chia lưới lên, lưới càng được chia nhỏ, số lượng pixel càng nhiều, thì bức ảnh kỹ thuật số càng chi tiết và xấp xỉ càng sát với hình ảnh trong thế giới thực.") as tracker:
            self.update_subtitle("Nếu tăng số ô chia lưới, số lượng pixel càng nhiều,\nthì bức ảnh càng chi tiết và sát với thực tế.")
            self.play(
                ReplacementTransform(step2_group, res_group),
                FadeOut(numbers_group), FadeOut(left_bracket), FadeOut(right_bracket)
            )
            self.play(subdivide_anim(grid_8x8, grid_16x16), run_time=1.5)
            self.play(subdivide_anim(grid_16x16, grid_32x32), run_time=1.5)

        with self.voiceover(text="Đó chính là lý do ta gọi số lượng ô chiều ngang và chiều dọc là độ phân giải.") as tracker:
            self.update_subtitle("Đó chính là lý do ta gọi số lượng ô chiều ngang\nvà chiều dọc là Độ phân giải.")
            img_64x64 = create_pixel_image(64, 64)
            vgroup_32x32 = flatten_2d(grid_32x32)
            self.play(FadeOut(vgroup_32x32), FadeIn(img_64x64), run_time=0.5)
            
            current_img_mob = img_64x64
            max_res = min(img_w // 2, img_h // 2)
            curr_val = 128
            
            while curr_val <= max_res:
                next_img_mob = create_pixel_image(curr_val, curr_val)
                self.play(FadeIn(next_img_mob), run_time=0.3)
                self.remove(current_img_mob)
                current_img_mob = next_img_mob
                curr_val *= 2
                
            if curr_val / 2 != max_res:
                final_img_mob = create_pixel_image(max_res, max_res)
                self.play(FadeIn(final_img_mob), run_time=0.3)
                self.remove(current_img_mob)

Manim Community v0.19.0

[05/09/26 06:47:14] WARNING  Some options were not used: {'shortest': '1', 'metadata':     ]8;id=524040;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=665999;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#801\801]8;;\
                             'comment=Rendered with Manim Community v0.19.0'}                                      

In [33]:
%%manim -v ERROR --resolution 720,1280 --frame_rate 15 --flush_cache \
        --media_dir /Users/doanvinhnhan/Roo-Code/media RGBTensorScene
"""
PHẦN CÒN LẠI CỦA VIDEO: "Bức ảnh trong máy tính thực chất là gì?"
Bao gồm: Phần 2 (RGB/Tensor), Phần 3 (JPG/DCT), Phần 4 (PNG/DEFLATE)

Chạy toàn bộ video (ghép Phần 1 vào):
    %%manim -v ERROR --resolution 720,1280 --frame_rate 15 --flush_cache \
            --media_dir /Users/doanvinhnhan/Roo-Code/media FullImageVideo

Hoặc render riêng từng scene:
    %%manim -v ERROR --resolution 720,1280 --frame_rate 15 --flush_cache \
            --media_dir /Users/doanvinhnhan/Roo-Code/media RGBTensorScene
"""

import os, sys
import numpy as np
from PIL import Image

PROJECT_ROOT = '/Users/doanvinhnhan/Roo-Code'
if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from manim import *

try:
    from skills.fami_lib import FaMIBaseScene
except ImportError:
    from manim_voiceover import VoiceoverScene
    FaMIBaseScene = VoiceoverScene

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# HELPER UTILITIES dùng chung toàn bộ video
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

IMG_PATH   = "/Users/doanvinhnhan/Roo-Code/assets/Monalisa.jpg"
MANIM_H    = 4.5
IMAGE_SHIFT = UP * 0.5


def load_image():
    pil_img = Image.open(IMG_PATH)
    return pil_img, pil_img.size


def make_channel_matrix(pil_img, channel_idx: int, rows=8, cols=8):
    """Trả về VGroup lưới ô màu cho một kênh R/G/B."""
    arr = np.array(pil_img.convert('RGB'))
    img_h, img_w = arr.shape[:2]
    aspect = img_w / img_h
    mat_h = MANIM_H * 0.8
    mat_w = mat_h * aspect
    cell_h = mat_h / rows
    cell_w = mat_w / cols

    CHANNEL_COLORS = [
        lambda v: rgb_to_color([v/255, 0, 0]),   # R
        lambda v: rgb_to_color([0, v/255, 0]),   # G
        lambda v: rgb_to_color([0, 0, v/255]),   # B
    ]
    cells = VGroup()
    values = []
    for i in range(rows):
        for j in range(cols):
            r0 = int(i * img_h / rows);  r1 = max(r0+1, int((i+1)*img_h/rows))
            c0 = int(j * img_w / cols);  c1 = max(c0+1, int((j+1)*img_w/cols))
            val = int(np.mean(arr[r0:r1, c0:c1, channel_idx]))
            values.append(val)
            rect = Rectangle(width=cell_w, height=cell_h)
            rect.set_fill(CHANNEL_COLORS[channel_idx](val), opacity=1)
            rect.set_stroke(color=WHITE, width=0.4)
            x = -mat_w/2 + (j + 0.5)*cell_w
            y =  mat_h/2 - (i + 0.5)*cell_h
            rect.move_to([x, y, 0])
            cells.add(rect)
    return cells, values, (mat_w, mat_h)


def make_number_overlay(cells, values, font_size=9):
    """Đặt số lên từng ô màu."""
    nums = VGroup()
    for rect, val in zip(cells, values):
        t = Text(str(val), font_size=font_size)
        t.set_color(BLACK if val > 160 else WHITE)
        t.move_to(rect.get_center())
        nums.add(t)
    return nums


def make_bracket(target, direction=LEFT):
    brace = Brace(target, direction=direction)
    return brace


def heat_color(frac):
    """0→dark blue, 0.5→yellow, 1→bright white."""
    frac = max(0.0, min(1.0, frac))
    if frac < 0.5:
        t = frac * 2
        return rgb_to_color([t, t, 0.8*(1-t)])
    else:
        t = (frac - 0.5) * 2
        return rgb_to_color([0.8 + 0.2*t, 0.8 + 0.2*t, t])


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PHẦN 2: RGB TENSOR & DỮ LIỆU
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RGBTensorScene(FaMIBaseScene):
    def construct(self):
        pil_img, (img_w, img_h) = load_image()

        # ── Tiêu đề phần ────────────────────────────────────────────────────
        part_title = Text("PHẦN 2: THÊM MÀU SẮC", font_size=36, color=YELLOW,
                          weight=BOLD).to_edge(UP, buff=0.25)

        with self.voiceover("Tiếp theo, ta sẽ cùng đi phủ màu cho hình ảnh này.") as tr:
            self.update_subtitle("Tiếp theo, ta sẽ phủ màu cho hình ảnh này.")
            self.play(Write(part_title))

        # ── Xây dựng 3 ma trận kênh màu ─────────────────────────────────────
        LABELS = ["R", "G", "B"]
        LABEL_COLORS = [RED, GREEN, BLUE]
        OFFSETS = [LEFT*3.5, ORIGIN, RIGHT*3.5]

        channel_groups = []
        for ch_idx in range(3):
            cells, vals, (mw, mh) = make_channel_matrix(pil_img, ch_idx, 8, 8)
            cells.move_to(OFFSETS[ch_idx] + DOWN*0.3)
            nums = make_number_overlay(cells, vals, font_size=7)
            label = Text(LABELS[ch_idx], font_size=40, color=LABEL_COLORS[ch_idx],
                         weight=BOLD)
            label.next_to(cells, UP, buff=0.15)
            frame = SurroundingRectangle(cells, color=LABEL_COLORS[ch_idx],
                                         buff=0.05, stroke_width=2)
            channel_groups.append(VGroup(cells, nums, label, frame))

        # ── Bắt đầu từ ảnh đen trắng ở phần 1 ──────────────────────────────
        gray_cells, gray_vals, (mw_g, mh_g) = make_channel_matrix(
            pil_img.convert('L').convert('RGB'), 0, 8, 8)
        gray_cells.move_to(ORIGIN + DOWN*0.3)
        gray_nums = make_number_overlay(gray_cells, gray_vals, font_size=7)
        bw_label = Text("Ảnh xám f(x,y)", font_size=28, color=GRAY_B)
        bw_label.next_to(gray_cells, UP, buff=0.2)

        with self.voiceover("Về mặt toán học, thay vì một ma trận 2 chiều đơn lẻ,\
 một bức ảnh màu là một cấu trúc dữ liệu ba chiều – một Tensor.") as tr:
            self.update_subtitle("Ảnh màu là một cấu trúc dữ liệu ba chiều – một Tensor.")
            self.play(FadeIn(gray_cells), FadeIn(gray_nums), Write(bw_label))

        # Tách ra thành 3 kênh
        with self.voiceover("Nó bao gồm 3 ma trận xếp chồng lên nhau tương ứng với\
 ba kênh màu Đỏ, Xanh lục và Xanh lam – không gian màu RGB.") as tr:
            self.update_subtitle("3 ma trận tương ứng 3 kênh màu: Đỏ, Xanh lục, Xanh lam – RGB.")
            self.play(
                FadeOut(bw_label),
                ReplacementTransform(gray_cells.copy(), channel_groups[0][0]),
                ReplacementTransform(gray_cells.copy(), channel_groups[1][0]),
                ReplacementTransform(gray_cells,        channel_groups[2][0]),
                FadeOut(gray_nums),
                run_time=1.5
            )
            for grp in channel_groups:
                self.play(
                    FadeIn(grp[1]),  # nums
                    Write(grp[2]),   # label
                    Create(grp[3]),  # frame
                    run_time=0.5
                )

        # ── Tạo hiệu ứng 3D xếp chồng (giả 3D bằng shift + opacity) ─────────
        STACK_OFFSET = np.array([0.18, -0.12, 0])  # shift per layer
        tensor_label = MathTex(r"\mathbf{T} \in \mathbb{R}^{H \times W \times 3}",
                               font_size=32, color=WHITE)
        tensor_label.to_edge(DOWN, buff=0.6)

        with self.voiceover("Sự phối hợp cường độ của 3 kênh này tại cùng một tọa độ\
 x,y sẽ đánh lừa tế bào hình nón trong mắt bạn, tạo ra cảm giác về hàng triệu màu sắc.") as tr:
            self.update_subtitle("Phối hợp R, G, B đánh lừa tế bào hình nón mắt người,\ntạo hàng triệu màu sắc.")
            # Trượt 3 lớp lại gần nhau rồi xếp chồng giả 3D
            self.play(
                channel_groups[0].animate.move_to(ORIGIN + DOWN*0.3 + STACK_OFFSET*2),
                channel_groups[1].animate.move_to(ORIGIN + DOWN*0.3 + STACK_OFFSET*1),
                channel_groups[2].animate.move_to(ORIGIN + DOWN*0.3 + STACK_OFFSET*0),
                run_time=1.5
            )
            self.play(Write(tensor_label))

        # Thêm mũi tên và nhãn "Tensor 3D"
        tensor_brace = Brace(
            VGroup(channel_groups[0], channel_groups[2]), LEFT, color=WHITE)
        tensor_brace_label = tensor_brace.get_text("Chiều sâu = 3", font_size=20)

        with self.voiceover("Khối dữ liệu ba chiều này được gọi là Tensor – nền tảng\
 của mọi ảnh màu kỹ thuật số.") as tr:
            self.update_subtitle("Khối dữ liệu 3 chiều này gọi là Tensor –\nnền tảng của mọi ảnh màu kỹ thuật số.")
            self.play(Create(tensor_brace), Write(tensor_brace_label))

        # ── Nghịch lý dữ liệu ───────────────────────────────────────────────
        self.play(
            FadeOut(tensor_brace, tensor_brace_label, tensor_label),
            *[FadeOut(g) for g in channel_groups],
            run_time=0.8
        )

        # Tính toán thực tế
        W, H = 2880, 1800
        raw_MB = W * H * 3 / 1e6           # ~15.55 MB thực ra
        raw_MB_display = 47.0              # dùng con số kịch bản
        fps_60_GB = raw_MB_display * 60 / 1024

        size_formula = MathTex(
            r"2880 \times 1800 \times 3 \text{ bytes}",
            r"= 15{,}552{,}000 \text{ bytes}",
            r"\approx 47 \text{ MB}",
            font_size=28
        ).arrange(DOWN, buff=0.15).move_to(UP*1.5)

        counter_label = Text("Kích thước ảnh đơn:", font_size=26, color=GRAY_A)
        counter_val = DecimalNumber(0, num_decimal_places=1,
                                   font_size=52, color=GREEN_C)
        counter_unit = Text(" MB", font_size=40, color=GREEN_C)
        counter_row = VGroup(counter_val, counter_unit).arrange(RIGHT, buff=0.05)
        counter_block = VGroup(counter_label, counter_row).arrange(DOWN, buff=0.2)
        counter_block.next_to(size_formula, DOWN, buff=0.5)

        video_warn = Text("1 giây video 60fps @ 2880×1800  ≈  2.75 GB  🔥",
                          font_size=24, color=RED)
        video_warn.next_to(counter_block, DOWN, buff=0.5)

        with self.voiceover("Tuy nhiên, nếu giữ nguyên một bức ảnh độ phân giải\
 2880 nhân 1800 ở dạng ma trận RGB nguyên bản, nó đã có kích thước xấp xỉ 47MB.") as tr:
            self.update_subtitle("Ảnh 2880×1800 RGB nguyên bản ≈ 47 MB,\nchưa qua bất kỳ nén nào.")
            self.play(Write(size_formula[0]), run_time=0.8)
            self.play(Write(size_formula[1]), run_time=0.8)
            self.play(Write(size_formula[2]), run_time=0.8)
            self.play(FadeIn(counter_block))
            self.play(
                counter_val.animate.set_value(raw_MB_display),
                UpdateFromFunc(counter_val,
                    lambda m: m.set_color(
                        RED if m.get_value() > 30 else
                        YELLOW if m.get_value() > 15 else GREEN_C
                    )
                ),
                run_time=2
            )

        with self.voiceover("Tức là một video 60fps dài một giây với độ phân giải\
 này sẽ nặng gần 3GB. Thật khủng khiếp!") as tr:
            self.update_subtitle("Video 60fps × 1 giây ≈ 2.75 GB.\nThật khủng khiếp!")
            self.play(Write(video_warn))
            self.play(Indicate(video_warn, scale_factor=1.1, color=RED), run_time=0.6)
            self.play(Indicate(video_warn, scale_factor=1.1, color=RED), run_time=0.6)

        # ── Giải pháp: JPG & PNG ─────────────────────────────────────────────
        solution_text = Text("Giải pháp: Nén ảnh – JPG & PNG", font_size=36,
                             color=YELLOW, weight=BOLD)
        solution_text.next_to(video_warn, DOWN, buff=0.5)

        jpg_box = RoundedRectangle(width=2.2, height=1.2, corner_radius=0.2,
                                   color=ORANGE, fill_opacity=0.15)
        jpg_label = Text("JPG\nNén tổn hao", font_size=22, color=ORANGE)
        jpg_group = VGroup(jpg_box, jpg_label).arrange_in_grid(1, 1)
        jpg_label.move_to(jpg_box.get_center())

        png_box = RoundedRectangle(width=2.2, height=1.2, corner_radius=0.2,
                                   color=BLUE, fill_opacity=0.15)
        png_label = Text("PNG\nKhông tổn hao", font_size=22, color=BLUE)
        png_label.move_to(png_box.get_center())
        png_group = VGroup(png_box, png_label)

        vs_text = Text("VS", font_size=36, color=GRAY_A, weight=BOLD)
        formats_row = VGroup(jpg_group, vs_text, png_group).arrange(RIGHT, buff=0.5)
        formats_row.next_to(solution_text, DOWN, buff=0.35)

        with self.voiceover("Đó là lý do các tiêu chuẩn nén ảnh như JPG và PNG ra\
 đời. Chúng đại diện cho hai trường phái toán học hoàn toàn trái ngược nhau!") as tr:
            self.update_subtitle("JPG và PNG – hai trường phái toán học\nhoàn toàn trái ngược nhau!")
            self.play(
                FadeOut(size_formula, counter_block, video_warn),
                Write(solution_text)
            )
            self.play(FadeIn(jpg_group), Write(vs_text), FadeIn(png_group))
            self.play(Indicate(jpg_group, color=ORANGE), Indicate(png_group, color=BLUE))

        self.play(*[FadeOut(m) for m in self.mobjects])


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PHẦN 3: JPG – DCT & LƯỢNG TỬ HÓA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class JPGScene(FaMIBaseScene):
    def construct(self):
        pil_img, (img_w, img_h) = load_image()

        # ── Tiêu đề ──────────────────────────────────────────────────────────
        part_title = Text("PHẦN 3: JPG – BẬC THẦY ĐÁNH LỪA THỊ GIÁC",
                          font_size=28, color=ORANGE, weight=BOLD).to_edge(UP, buff=0.2)

        with self.voiceover("Đầu tiên là JPG – đại diện cho cơ chế nén tổn hao.") as tr:
            self.update_subtitle("JPG – đại diện cho cơ chế nén tổn hao (lossy compression).")
            self.play(Write(part_title))

        # ━━ 3A: CHUYỂN ĐỔI YCbCr ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

        # Biểu diễn 3 kênh RGB bên trái
        rgb_rects = VGroup(
            Rectangle(width=1.6, height=0.9, fill_color=RED,   fill_opacity=0.7, stroke_color=WHITE, stroke_width=1),
            Rectangle(width=1.6, height=0.9, fill_color=GREEN, fill_opacity=0.7, stroke_color=WHITE, stroke_width=1),
            Rectangle(width=1.6, height=0.9, fill_color=BLUE,  fill_opacity=0.7, stroke_color=WHITE, stroke_width=1),
        ).arrange(DOWN, buff=0.1).move_to(LEFT*4 + DOWN*0.3)
        rgb_labels = VGroup(*[
            Text(t, font_size=22, color=c).move_to(r.get_center())
            for t, c, r in zip(["R", "G", "B"], [WHITE, WHITE, WHITE], rgb_rects)
        ])
        rgb_group = VGroup(rgb_rects, rgb_labels)
        rgb_title = Text("RGB", font_size=28, color=WHITE, weight=BOLD)
        rgb_title.next_to(rgb_rects, UP, buff=0.15)

        arrow_rgb_ycbcr = Arrow(LEFT*2.5, LEFT*1.0, buff=0, color=YELLOW)

        # Biểu diễn 3 kênh YCbCr bên phải
        ycbcr_colors = [GRAY_A, "#4488FF", "#FF4488"]
        ycbcr_labels_txt = ["Y\n(Luma)", "Cb\n(Sắc độ lam)", "Cr\n(Sắc độ đỏ)"]
        ycbcr_rects = VGroup()
        ycbcr_txt = VGroup()
        ycbcr_sizes = [(1.6, 0.9), (0.8, 0.9), (0.8, 0.9)]  # Cb & Cr nhỏ hơn
        for i, (c, lbl, sz) in enumerate(zip(ycbcr_colors, ycbcr_labels_txt, ycbcr_sizes)):
            r = Rectangle(width=sz[0], height=sz[1], fill_color=c,
                          fill_opacity=0.7, stroke_color=WHITE, stroke_width=1)
            ycbcr_rects.add(r)
            t = Text(lbl, font_size=14, color=WHITE).move_to(r.get_center())
            ycbcr_txt.add(t)
        ycbcr_rects.arrange(DOWN, buff=0.1).move_to(RIGHT*2 + DOWN*0.3)
        for t, r in zip(ycbcr_txt, ycbcr_rects):
            t.move_to(r.get_center())
        ycbcr_title = Text("YCbCr", font_size=28, color=YELLOW, weight=BOLD)
        ycbcr_title.next_to(ycbcr_rects, UP, buff=0.15)

        # Phương trình biến đổi
        eq_Y  = MathTex(r"Y  &= 0.257R + 0.504G + 0.098B + 16",  font_size=22, color=GRAY_A)
        eq_Cb = MathTex(r"Cb &= -0.148R - 0.291G + 0.439B + 128", font_size=22, color="#4488FF")
        eq_Cr = MathTex(r"Cr &= 0.439R - 0.368G - 0.071B + 128",  font_size=22, color="#FF4488")
        eq_group = VGroup(eq_Y, eq_Cb, eq_Cr).arrange(DOWN, aligned_edge=LEFT, buff=0.2)
        eq_group.move_to(DOWN*2.5)

        with self.voiceover("Thuật toán này dựa trên một lỗ hổng sinh lý của mắt người:\
 Võng mạc cực kỳ nhạy cảm với độ sáng, nhưng lại kém nhận diện chi tiết màu sắc nhỏ.") as tr:
            self.update_subtitle("Mắt người nhạy với độ sáng nhưng kém nhận diện\nchi tiết màu sắc nhỏ bé.")
            self.play(FadeIn(rgb_group), Write(rgb_title))

        with self.voiceover("JPG bóc tách không gian RGB sang không gian màu YCbCr\
 bằng hệ phương trình ma trận tuyến tính.") as tr:
            self.update_subtitle("JPG chuyển RGB → YCbCr bằng hệ phương trình tuyến tính.")
            self.play(Write(eq_group), run_time=1.5)
            self.play(GrowArrow(arrow_rgb_ycbcr))
            self.play(Create(ycbcr_rects), Write(ycbcr_txt), Write(ycbcr_title))

        # Loại bỏ một nửa Cb và Cr (Chroma subsampling)
        strike_Cb = Line(ycbcr_rects[1].get_corner(UL),
                         ycbcr_rects[1].get_corner(DR), color=RED, stroke_width=3)
        strike_Cr = Line(ycbcr_rects[2].get_corner(UL),
                         ycbcr_rects[2].get_corner(DR), color=RED, stroke_width=3)
        half_label = Text("Vứt bỏ 50% dữ liệu sắc độ!", font_size=22, color=RED)
        half_label.next_to(ycbcr_rects, RIGHT, buff=0.3)

        with self.voiceover("Lớp cường độ sáng Y được giữ nguyên, còn dữ liệu ở hai\
 lớp sắc độ Cb và Cr bị hệ thống thẳng tay loại bỏ đi một nửa.") as tr:
            self.update_subtitle("Lớp Y giữ nguyên.\nCb và Cr bị loại bỏ 50% dữ liệu sắc độ!")
            self.play(
                ycbcr_rects[0].animate.set_fill(opacity=1.0),
                ycbcr_rects[1].animate.set_fill(opacity=0.3),
                ycbcr_rects[2].animate.set_fill(opacity=0.3),
                run_time=1
            )
            self.play(Create(strike_Cb), Create(strike_Cr))
            self.play(Write(half_label))

        self.play(*[FadeOut(m) for m in self.mobjects if m is not part_title])
        self.play(part_title.animate.to_edge(UP, buff=0.2))

        # ━━ 3B: BIẾN ĐỔI DCT TRÊN KHỐI 8×8 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

        # Lấy một khối 8×8 thực tế từ ảnh
        arr_gray = np.array(pil_img.convert('L'), dtype=float)
        # Lấy khối ở trung tâm ảnh
        cy, cx = arr_gray.shape[0]//2, arr_gray.shape[1]//2
        block_orig = arr_gray[cy-4:cy+4, cx-4:cx+4].copy()

        # Tính DCT 2D thực tế
        from scipy.fft import dctn
        block_dct = dctn(block_orig - 128, norm='ortho')

        CELL = 0.65  # kích thước ô vuông trong Manim

        def make_8x8_grid(values_2d, cell_size=CELL, colormap_fn=None, num_fmt="d",
                          font_sz=12, label_color_fn=None):
            """Trả về (grid_cells VGroup, grid_nums VGroup)"""
            cells = VGroup()
            nums  = VGroup()
            vmin, vmax = values_2d.min(), values_2d.max()
            for i in range(8):
                for j in range(8):
                    v = values_2d[i, j]
                    if colormap_fn:
                        frac = (v - vmin) / (vmax - vmin + 1e-9)
                        fc = colormap_fn(frac)
                    else:
                        frac = (v - vmin) / (vmax - vmin + 1e-9)
                        fc = rgb_to_color([frac, frac, frac])
                    rect = Rectangle(width=cell_size, height=cell_size)
                    rect.set_fill(fc, opacity=1)
                    rect.set_stroke(color="#333333", width=0.5)
                    rect.move_to(
                        RIGHT*(j - 3.5)*cell_size + UP*(3.5 - i)*cell_size
                    )
                    cells.add(rect)
                    if num_fmt == "d":
                        s = str(int(round(v)))
                    else:
                        s = f"{v:.0f}"
                    lc = WHITE
                    if label_color_fn:
                        lc = label_color_fn(v)
                    t = Text(s, font_size=font_sz, color=lc)
                    t.move_to(rect.get_center())
                    nums.add(t)
            return cells, nums

        # Lưới không gian
        spatial_cells, spatial_nums = make_8x8_grid(
            block_orig,
            colormap_fn=lambda f: rgb_to_color([f, f, f]),
            label_color_fn=lambda v: BLACK if v > 128 else WHITE,
            font_sz=12
        )
        spatial_group = VGroup(spatial_cells, spatial_nums)
        spatial_group.move_to(LEFT*2.8 + DOWN*0.5)

        spatial_title = Text("Miền không gian f(x,y)", font_size=22, color=WHITE)
        spatial_title.next_to(spatial_cells, UP, buff=0.2)

        dct_label_title = Text("2D-DCT", font_size=28, color=YELLOW, weight=BOLD)
        dct_arrow = Arrow(LEFT*0.3 + DOWN*0.5, RIGHT*0.8 + DOWN*0.5,
                          buff=0.1, color=YELLOW, stroke_width=3)
        dct_label_title.next_to(dct_arrow, UP, buff=0.1)

        # Lưới tần số (dùng heatmap)
        freq_cells, freq_nums = make_8x8_grid(
            block_dct,
            colormap_fn=heat_color,
            label_color_fn=lambda v: BLACK if abs(v) < 50 else WHITE,
            font_sz=10
        )
        freq_group = VGroup(freq_cells, freq_nums)
        freq_group.move_to(RIGHT*2.8 + DOWN*0.5)
        freq_title = Text("Miền tần số F(u,v)", font_size=22, color=YELLOW)
        freq_title.next_to(freq_cells, UP, buff=0.2)

        dct_formula = MathTex(
            r"F(u,v) = \alpha(u)\alpha(v)\sum_{x}\sum_{y} f(x,y)"
            r"\cos\!\left[\frac{(2x+1)u\pi}{16}\right]\cos\!\left[\frac{(2y+1)v\pi}{16}\right]",
            font_size=18
        ).to_edge(DOWN, buff=0.5)

        with self.voiceover("Bức ảnh sẽ được băm thành các khối ma trận nhỏ 8 nhân 8.\
 Đây là một khối điển hình trong miền không gian.") as tr:
            self.update_subtitle("Bức ảnh được băm thành các khối 8×8.\nĐây là một khối trong miền không gian.")
            self.play(FadeIn(spatial_cells), Write(spatial_title))
            self.play(FadeIn(spatial_nums))

        with self.voiceover("Phương trình 2D-DCT sẽ chuyển đổi tín hiệu từ miền không\
 gian sang miền phổ tần số F(u,v).") as tr:
            self.update_subtitle("2D-DCT chuyển tín hiệu từ miền không gian\nsang miền phổ tần số F(u,v).")
            self.play(Write(dct_formula), run_time=1)
            self.play(GrowArrow(dct_arrow), Write(dct_label_title))
            # Hoạt ảnh: số trong ô spatial "lật" sang freq
            self.play(
                ReplacementTransform(spatial_cells.copy(), freq_cells),
                ReplacementTransform(spatial_nums.copy(),  freq_nums),
                Write(freq_title),
                run_time=2
            )

        # Highlight góc trên trái (tần số thấp / năng lượng cao)
        lo_freq_highlight = SurroundingRectangle(
            VGroup(*[freq_cells[i*8 + j] for i in range(3) for j in range(3)]),
            color=WHITE, buff=0.05, stroke_width=2
        )
        lo_freq_label = Text("Năng lượng cao\n(Tần số thấp)", font_size=16, color=WHITE)
        lo_freq_label.next_to(lo_freq_highlight, LEFT, buff=0.2)

        hi_freq_highlight = SurroundingRectangle(
            VGroup(*[freq_cells[i*8 + j] for i in range(5, 8) for j in range(5, 8)]),
            color=RED, buff=0.05, stroke_width=2
        )
        hi_freq_label = Text("Nhiễu / Chi tiết\n(Tần số cao)", font_size=16, color=RED)
        hi_freq_label.next_to(hi_freq_highlight, RIGHT, buff=0.15)

        with self.voiceover("Hiện tượng tuyệt mỹ gọi là 'Hội tụ năng lượng': Thông tin\
 quan trọng nhất dồn về góc trên trái, còn các nhiễu hạt chi tiết nhỏ bị đẩy sang góc phải.") as tr:
            self.update_subtitle("'Hội tụ năng lượng': Thông tin quan trọng ở góc trên trái,\nnhiễu hạt ở góc phải.")
            self.play(Create(lo_freq_highlight), Write(lo_freq_label))
            self.play(Create(hi_freq_highlight), Write(hi_freq_label))
            self.play(
                Indicate(lo_freq_highlight, scale_factor=1.05, color=WHITE),
                Indicate(hi_freq_highlight, scale_factor=1.05, color=RED),
                run_time=0.8
            )

        # ━━ 3C: LƯỢNG TỬ HÓA ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

        self.play(
            *[FadeOut(m) for m in [
                spatial_group, spatial_title, dct_arrow, dct_label_title,
                lo_freq_highlight, lo_freq_label, hi_freq_highlight, hi_freq_label,
                dct_formula
            ]]
        )
        freq_group.move_to(LEFT*2.5 + DOWN*0.5)
        freq_title.next_to(freq_cells, UP, buff=0.2)

        # Ma trận Q chuẩn JPEG cho vùng Luma
        Q_standard = np.array([
            [16, 11, 10, 16, 24,  40,  51,  61],
            [12, 12, 14, 19, 26,  58,  60,  55],
            [14, 13, 16, 24, 40,  57,  69,  56],
            [14, 17, 22, 29, 51,  87,  80,  62],
            [18, 22, 37, 56, 68, 109, 103,  77],
            [24, 35, 55, 64, 81, 104, 113,  92],
            [49, 64, 78, 87,103, 121, 120, 101],
            [72, 92, 95, 98,112, 100, 103,  99],
        ], dtype=float)

        q_cells, q_nums = make_8x8_grid(
            Q_standard,
            colormap_fn=lambda f: interpolate_color(DARK_BLUE, RED, f),
            label_color_fn=lambda v: WHITE,
            font_sz=9
        )
        q_group = VGroup(q_cells, q_nums)
        q_group.move_to(RIGHT*0.3 + DOWN*0.5)
        q_title = Text("Ma trận lượng tử hóa Q", font_size=20, color=RED)
        q_title.next_to(q_cells, UP, buff=0.2)

        divide_sign = MathTex(r"\div", font_size=36, color=YELLOW).move_to(
            (freq_cells.get_right() + q_cells.get_left()) / 2 + DOWN*0.5
        )

        # Kết quả sau lượng tử hóa
        block_quantized = np.round(block_dct / Q_standard).astype(int)
        qout_cells, qout_nums = make_8x8_grid(
            block_quantized.astype(float),
            colormap_fn=lambda f: rgb_to_color([0.1, 0.1, 0.15]) if abs(
                (f * (block_quantized.max()-block_quantized.min()) + block_quantized.min())) < 2
            else interpolate_color(DARK_BLUE, YELLOW, f),
            label_color_fn=lambda v: RED if v == 0 else WHITE,
            font_sz=11
        )
        qout_group = VGroup(qout_cells, qout_nums)
        qout_group.move_to(RIGHT*4.5 + DOWN*0.5)
        qout_title = Text("Sau lượng tử hóa", font_size=20, color=GREEN_C)
        qout_title.next_to(qout_cells, UP, buff=0.2)

        equal_sign = MathTex(r"\approx", font_size=36, color=YELLOW).move_to(
            (q_cells.get_right() + qout_cells.get_left()) / 2 + DOWN*0.5
        )

        quant_formula = MathTex(
            r"\hat{F}(u,v) = \text{round}\!\left(\frac{F(u,v)}{Q(u,v)}\right)",
            font_size=24
        ).to_edge(DOWN, buff=0.4)

        with self.voiceover("Tiếp đó, JPG kích hoạt cỗ máy hủy diệt mang tên Lượng\
 tử hóa: Lấy các hệ số tần số chia cho ma trận Q rồi làm tròn.") as tr:
            self.update_subtitle("Lượng tử hóa: Chia F(u,v) cho ma trận Q rồi làm tròn.")
            self.play(Write(quant_formula))
            self.play(
                FadeIn(q_cells), Write(q_title), FadeIn(q_nums),
                Write(divide_sign)
            )

        # Hiệu ứng: các số ở góc dưới phải → 0 đỏ
        zeros_in_block = [(i, j) for i in range(5, 8) for j in range(5, 8)
                          if block_quantized[i, j] == 0]
        with self.voiceover("Kết quả: hàng loạt dữ liệu tần số cao bị nghiền nát\
 thành các số 0 tròn trĩnh. Chính chuỗi số 0 này giúp dung lượng file co ngót hàng chục lần.") as tr:
            self.update_subtitle("Hàng loạt dữ liệu bị nghiền thành số 0.\nChuỗi số 0 dài giúp nén file hàng chục lần.")
            self.play(Write(equal_sign))
            self.play(FadeIn(qout_cells), FadeIn(qout_nums), Write(qout_title), run_time=1)
            # Flash vào các ô bằng 0
            zero_rects = VGroup(*[
                qout_cells[i*8 + j]
                for i in range(8) for j in range(8)
                if block_quantized[i, j] == 0
            ])
            self.play(Flash(zero_rects, color=RED, line_length=0.1, flash_radius=0.15),
                      run_time=1)

        # ━━ 3D: MSE – SỰ TỔN HAO ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

        self.play(*[FadeOut(m) for m in self.mobjects if m is not part_title])

        # Khôi phục và so sánh ảnh gốc vs ảnh giải nén
        from scipy.fft import idctn
        block_reconstructed = idctn(block_quantized * Q_standard, norm='ortho') + 128
        block_reconstructed = np.clip(block_reconstructed, 0, 255)

        diff_block = (block_orig - block_reconstructed)
        mse_val = float(np.mean(diff_block**2))

        orig_cells, orig_nums = make_8x8_grid(
            block_orig, colormap_fn=lambda f: rgb_to_color([f, f, f]),
            label_color_fn=lambda v: BLACK if v > 128 else WHITE, font_sz=11
        )
        orig_cells.move_to(LEFT*4 + UP*0.5)
        orig_nums.move_to(LEFT*4 + UP*0.5)
        orig_title = Text("Ảnh gốc I", font_size=22, color=WHITE)
        orig_title.next_to(orig_cells, UP, buff=0.15)

        minus_sign = MathTex(r"-", font_size=40, color=WHITE).move_to(LEFT*2)

        recon_cells, recon_nums = make_8x8_grid(
            block_reconstructed,
            colormap_fn=lambda f: rgb_to_color([f, f, f]),
            label_color_fn=lambda v: BLACK if v > 128 else WHITE, font_sz=11
        )
        recon_cells.move_to(LEFT*0.5 + UP*0.5)
        recon_nums.move_to(LEFT*0.5 + UP*0.5)
        recon_title = Text("Ảnh giải nén K", font_size=22, color=ORANGE)
        recon_title.next_to(recon_cells, UP, buff=0.15)

        eq_sign = MathTex(r"=", font_size=40, color=WHITE).move_to(RIGHT*1.5)

        # Diff map
        diff_cells, diff_nums = make_8x8_grid(
            np.abs(diff_block),
            colormap_fn=lambda f: interpolate_color(BLACK, RED, f**0.5),
            label_color_fn=lambda v: WHITE, font_sz=11
        )
        diff_cells.move_to(RIGHT*3 + UP*0.5)
        diff_title = Text("Ma trận sai lệch |I−K|", font_size=22, color=RED)
        diff_title.next_to(diff_cells, UP, buff=0.15)

        mse_display = MathTex(
            r"MSE = \frac{1}{mn}\sum_{i,j}[I(i,j)-K(i,j)]^2 \;=\; " + f"{mse_val:.2f}",
            font_size=24, color=RED
        ).to_edge(DOWN, buff=0.5)

        with self.voiceover("Để đo lường sự biến dạng này, khoa học sử dụng công thức\
 Hệ số Sai số Trung bình Bình phương MSE giữa ảnh gốc và ảnh đã giải nén.") as tr:
            self.update_subtitle("MSE đo lường sự biến dạng giữa ảnh gốc I\nvà ảnh giải nén K.")
            self.play(FadeIn(orig_cells), FadeIn(orig_nums), Write(orig_title))
            self.play(Write(minus_sign))
            self.play(FadeIn(recon_cells), FadeIn(recon_nums), Write(recon_title))
            self.play(Write(eq_sign))
            self.play(FadeIn(diff_cells), Write(diff_title))
            self.play(FadeIn(diff_nums))
            self.play(Write(mse_display))

        with self.voiceover("Đổi lại, dữ liệu bị mất vĩnh viễn – đó chính là bản chất\
 của nén tổn hao.") as tr:
            self.update_subtitle("Dữ liệu bị mất vĩnh viễn –\nbản chất của nén tổn hao (lossy).")
            self.play(Indicate(mse_display, color=RED, scale_factor=1.1))

        self.play(*[FadeOut(m) for m in self.mobjects])


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PHẦN 4: PNG – NÉN KHÔNG TỔN HAO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class PNGScene(FaMIBaseScene):
    def construct(self):
        pil_img, (img_w, img_h) = load_image()

        part_title = Text("PHẦN 4: PNG – NÉN KHÔNG TỔN HAO",
                          font_size=28, color=BLUE, weight=BOLD).to_edge(UP, buff=0.2)

        with self.voiceover("PNG sử dụng cơ chế nén không tổn hao – bảo toàn nguyên\
 trạng từng bit dữ liệu.") as tr:
            self.update_subtitle("PNG – cơ chế nén không tổn hao (lossless compression).\nBảo toàn 100% dữ liệu.")
            self.play(Write(part_title))

        # ━━ 4A: MÀNG LỌC DỰ BÁO ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

        # Lấy một dải pixel thực tế từ một vùng bầu trời đơn điệu trong ảnh
        arr_gray = np.array(pil_img.convert('L'))
        sky_row_idx = int(arr_gray.shape[0] * 0.15)   # gần đỉnh ảnh
        sky_slice   = arr_gray[sky_row_idx, :8].astype(int)
        # Đảm bảo slice có đủ 8 pixels, fallback nếu cần
        if len(sky_slice) < 8:
            sky_slice = np.array([120, 122, 121, 123, 122, 124, 121, 123])

        PIXEL_W = 1.1
        pixel_boxes = VGroup()
        pixel_nums  = VGroup()
        pixel_vals  = list(sky_slice[:8])

        for j, v in enumerate(pixel_vals):
            box = Rectangle(width=PIXEL_W, height=PIXEL_W)
            brightness = v / 255.0
            box.set_fill(rgb_to_color([brightness]*3), opacity=1)
            box.set_stroke(WHITE, width=1)
            box.move_to(RIGHT * (j - 3.5) * PIXEL_W + UP * 1.5)
            pixel_boxes.add(box)
            t = Text(str(v), font_size=18,
                     color=BLACK if v > 128 else WHITE)
            t.move_to(box.get_center())
            pixel_nums.add(t)

        pixel_label = Text("Dải pixel thực tế (dòng bầu trời)", font_size=22, color=WHITE)
        pixel_label.next_to(pixel_boxes, UP, buff=0.2)

        with self.voiceover("Thay vì dùng phương trình tần số và cắt xén dữ liệu, PNG\
 phân tích cấu trúc bức ảnh bằng các Màng lọc nội suy dự báo.") as tr:
            self.update_subtitle("PNG dùng Màng lọc nội suy dự báo thay vì cắt xén dữ liệu.")
            self.play(FadeIn(pixel_boxes), Write(pixel_label))
            self.play(FadeIn(pixel_nums))

        # Highlight và tính Delta
        delta_formula = MathTex(
            r"\Delta = f(x,y) - f_{\text{predict}}(x,y)",
            font_size=30, color=BLUE
        ).next_to(pixel_boxes, DOWN, buff=0.5)

        predict_label_txt = Text("Dự báo (Sub filter): f_predict = pixel bên trái",
                                 font_size=19, color=GRAY_A)
        predict_label_txt.next_to(delta_formula, DOWN, buff=0.2)

        with self.voiceover("Nguyên lý của nó rất thanh lịch: Trong một mảng màu thực\
 tế, các pixel kề nhau thường có cường độ gần như bằng nhau.") as tr:
            self.update_subtitle("Các pixel kề nhau trong mảng màu đồng đều\nthường có cường độ gần bằng nhau.")
            self.play(Write(delta_formula))
            self.play(Write(predict_label_txt))

        # Tính delta thực tế (SUB filter: dự báo = pixel trước đó)
        deltas = [pixel_vals[0]] + [pixel_vals[j] - pixel_vals[j-1]
                                     for j in range(1, len(pixel_vals))]

        highlight_box = SurroundingRectangle(pixel_boxes[0], color=YELLOW, buff=0.05)
        delta_boxes   = VGroup()
        delta_nums    = VGroup()

        for j in range(len(pixel_vals)):
            d = deltas[j]
            box = Rectangle(width=PIXEL_W, height=PIXEL_W * 0.8)
            box.set_fill(GREEN if d == 0 else (BLUE if d > 0 else RED),
                         opacity=0.4 + 0.4 * min(abs(d)/10.0, 1.0))
            box.set_stroke(WHITE, width=0.8)
            box.move_to(RIGHT * (j - 3.5) * PIXEL_W + DOWN * 0.3)
            delta_boxes.add(box)
            sign = "+" if d > 0 else ""
            t = Text(f"{sign}{d}", font_size=18, color=GREEN_C if d == 0 else WHITE)
            t.move_to(box.get_center())
            delta_nums.add(t)

        delta_row_label = Text("Giá trị Delta Δ (rất nhỏ!)", font_size=20, color=GREEN_C)
        delta_row_label.next_to(delta_boxes, DOWN, buff=0.2)

        with self.voiceover("Thuật toán của PNG sẽ chạy phương trình phỏng đoán giá\
 trị của pixel tiếp theo. Sau đó, hệ thống chỉ lưu trữ hiệu số vi phân Delta giữa\
 giá trị thực tế và giá trị phỏng đoán.") as tr:
            self.update_subtitle("PNG chỉ lưu trữ hiệu số vi phân Δ = f(x,y) − f_predict(x,y),\nthay vì toàn bộ giá trị pixel.")
            self.play(Create(highlight_box))
            for j in range(len(pixel_vals)):
                self.play(
                    highlight_box.animate.move_to(pixel_boxes[j].get_center()),
                    FadeIn(delta_boxes[j]),
                    Write(delta_nums[j]),
                    run_time=0.35
                )
            self.play(Write(delta_row_label))

        self.play(FadeOut(highlight_box), FadeOut(predict_label_txt))

        # ━━ 4B: NGOÀI ĐẦU VÀO DEFLATE – LZ77 + HUFFMAN ━━━━━━━━━━━━━━━━━━━━━

        self.play(
            *[FadeOut(m) for m in [
                pixel_boxes, pixel_nums, pixel_label, delta_formula
            ]]
        )
        delta_boxes.generate_target()
        delta_nums.generate_target()
        delta_boxes.target.move_to(UP * 2.5)
        delta_nums.target.move_to(UP * 2.5)
        delta_row_label.generate_target()
        delta_row_label.target.next_to(delta_boxes.target, UP, buff=0.15)

        self.play(
            MoveToTarget(delta_boxes),
            MoveToTarget(delta_nums),
            MoveToTarget(delta_row_label),
        )

        # --- LZ77: Cửa sổ trượt ---
        # Biểu diễn một chuỗi ký tự lặp lại (mô phỏng)
        lz77_title = Text("Bước 1: LZ77 – Cửa sổ trượt tìm chuỗi lặp",
                          font_size=22, color=YELLOW).next_to(delta_boxes, DOWN, buff=0.45)

        byte_stream_vals = deltas * 2 + [0, 0, 0]   # giả lập chuỗi có lặp
        byte_boxes = VGroup()
        byte_texts = VGroup()
        for k, v in enumerate(byte_stream_vals[:12]):
            b = Rectangle(width=0.7, height=0.55)
            b.set_fill(DARK_GRAY, opacity=0.9)
            b.set_stroke(color=GRAY, width=0.8)
            b.move_to(RIGHT*(k - 5.5)*0.75 + DOWN*0.1)
            byte_boxes.add(b)
            s = str(v) if abs(v) < 10 else str(v)
            t = Text(s, font_size=14, color=WHITE)
            t.move_to(b.get_center())
            byte_texts.add(t)

        window_rect = SurroundingRectangle(
            VGroup(*byte_boxes[:4]), color=BLUE, buff=0.04, stroke_width=2
        )
        search_rect  = SurroundingRectangle(
            VGroup(*byte_boxes[4:8]), color=GREEN, buff=0.04, stroke_width=2
        )
        window_lbl = Text("Cửa sổ tìm kiếm", font_size=14, color=BLUE)
        window_lbl.next_to(window_rect, UP, buff=0.08)
        search_lbl = Text("Lookahead buffer", font_size=14, color=GREEN)
        search_lbl.next_to(search_rect, DOWN, buff=0.08)

        with self.voiceover("Sau đó, chuỗi số vi phân này được nén chặt bằng thuật\
 toán DEFLATE. Bước đầu tiên là LZ77: một cửa sổ trượt quét qua toàn bộ chuỗi\
 byte và tìm kiếm các đoạn lặp lại.") as tr:
            self.update_subtitle("DEFLATE bước 1: LZ77 dùng cửa sổ trượt\ntìm kiếm chuỗi byte lặp lại.")
            self.play(Write(lz77_title))
            self.play(FadeIn(byte_boxes), FadeIn(byte_texts))
            self.play(Create(window_rect), Write(window_lbl),
                      Create(search_rect), Write(search_lbl))
            # Trượt cửa sổ
            for shift in range(1, 4):
                self.play(
                    window_rect.animate.move_to(
                        VGroup(*byte_boxes[shift:shift+4]).get_center()
                        + UP*0.04
                    ),
                    search_rect.animate.move_to(
                        VGroup(*byte_boxes[shift+4:shift+8]).get_center()
                        + DOWN*0.04
                    ),
                    run_time=0.4
                )

        # Thay chuỗi lặp bằng back-reference (mô phỏng)
        ref_text = Text("(offset=8, length=3)", font_size=16, color=GREEN_C)
        ref_arrow = Arrow(
            byte_boxes[8].get_top() + UP*0.1,
            byte_boxes[0].get_top() + UP*0.1,
            buff=0.05, color=GREEN_C, stroke_width=1.5
        ).flip(UP).shift(UP*0.4)
        with self.voiceover("Các đoạn trùng lặp được thay thế bằng một cặp số nhỏ\
 gọn: khoảng cách và độ dài, thay vì sao chép toàn bộ chuỗi.") as tr:
            self.update_subtitle("Chuỗi lặp được thay bằng (offset, length),\ncực kỳ gọn nhẹ!")
            self.play(GrowArrow(ref_arrow), Write(ref_text))

        # --- Huffman Coding ---
        self.play(*[FadeOut(m) for m in [
            lz77_title, byte_boxes, byte_texts,
            window_rect, window_lbl, search_rect, search_lbl,
            ref_arrow, ref_text
        ]])

        huffman_title = Text("Bước 2: Huffman – Cây mã hóa xác suất",
                             font_size=22, color=YELLOW)
        huffman_title.next_to(delta_boxes, DOWN, buff=0.45)

        # Đếm tần suất các giá trị delta
        from collections import Counter
        freq_count = Counter(deltas)
        freq_sorted = sorted(freq_count.items(), key=lambda x: -x[1])[:5]

        # Vẽ cây Huffman đơn giản (4 lá)
        # Cấu trúc cây: root → hai nhánh, mỗi nhánh có 2 lá
        node_r = Circle(radius=0.28, color=WHITE, fill_opacity=0.15)
        node_r.move_to(ORIGIN + DOWN*0.2)
        root_lbl = Text("Root", font_size=14).move_to(node_r)

        level2_positions = [LEFT*2 + DOWN*1.2, RIGHT*2 + DOWN*1.2]
        level2_nodes = VGroup(*[
            Circle(radius=0.25, color=BLUE, fill_opacity=0.2).move_to(p)
            for p in level2_positions
        ])

        leaf_positions = [
            LEFT*3.2 + DOWN*2.4, LEFT*0.8 + DOWN*2.4,
            RIGHT*0.8 + DOWN*2.4, RIGHT*3.2 + DOWN*2.4
        ]
        leaf_vals = [freq_sorted[i][0] if i < len(freq_sorted) else 0
                     for i in range(4)]
        leaf_freqs = [freq_sorted[i][1] if i < len(freq_sorted) else 0
                      for i in range(4)]
        leaf_nodes = VGroup(*[
            Circle(radius=0.25, color=GREEN, fill_opacity=0.3).move_to(p)
            for p in leaf_positions
        ])
        leaf_labels = VGroup(*[
            Text(f"Δ={v}\n({f}×)", font_size=12).move_to(p)
            for v, f, p in zip(leaf_vals, leaf_freqs, leaf_positions)
        ])

        huffman_edges = VGroup()
        for i, lpos in enumerate(level2_positions):
            huffman_edges.add(Line(node_r.get_center(), lpos, color=GRAY_B, stroke_width=1.5))
        for i, lpos in enumerate(leaf_positions):
            parent_pos = level2_positions[i // 2]
            huffman_edges.add(Line(parent_pos, lpos, color=GRAY_B, stroke_width=1.5))

        # Mã Huffman: nhánh trái = 0, phải = 1
        bit_codes = ["00", "01", "10", "11"]
        code_labels = VGroup(*[
            Text(bc, font_size=14, color=GREEN_C).next_to(leaf_nodes[i], DOWN, buff=0.1)
            for i, bc in enumerate(bit_codes)
        ])
        # Nhãn nhánh 0/1
        branch_labels = VGroup()
        for i, (lpos, parent) in enumerate(zip(level2_positions, [node_r.get_center()]*2)):
            mid = (np.array(lpos) + np.array(parent)) / 2
            txt = Text("0" if i == 0 else "1", font_size=14, color=YELLOW)
            txt.move_to(mid + LEFT*0.2 if i == 0 else mid + RIGHT*0.2)
            branch_labels.add(txt)

        huffman_tree = VGroup(node_r, root_lbl, level2_nodes, leaf_nodes,
                              leaf_labels, huffman_edges, code_labels, branch_labels)
        huffman_tree.move_to(DOWN*1.4)

        with self.voiceover("Bước thứ hai của DEFLATE là mã hóa Huffman: Xây dựng\
 một cây nhị phân dựa trên tần suất xuất hiện của từng giá trị. Giá trị nào xuất\
 hiện nhiều nhất sẽ được gán mã bit ngắn nhất.") as tr:
            self.update_subtitle("Huffman: Giá trị xuất hiện nhiều → mã bit ngắn hơn.\nTối ưu hóa không gian lưu trữ.")
            self.play(Write(huffman_title))
            self.play(Create(huffman_edges), run_time=1)
            self.play(Create(node_r), Write(root_lbl))
            self.play(Create(level2_nodes), Write(branch_labels))
            self.play(Create(leaf_nodes), Write(leaf_labels))
            self.play(Write(code_labels))

        # ━━ 4C: KẾT QUẢ & TRONG SUỐT ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

        self.play(*[FadeOut(m) for m in self.mobjects if m is not part_title])

        # So sánh tỉ lệ nén
        comparison_title = Text("Kết quả: Nén không tổn hao", font_size=30,
                                 color=BLUE, weight=BOLD).next_to(part_title, DOWN, buff=0.35)

        # Thanh ngang biểu diễn kích thước
        BAR_FULL  = 6.0
        raw_bar   = Rectangle(width=BAR_FULL,    height=0.55, fill_color=RED,   fill_opacity=0.8)
        png_bar   = Rectangle(width=BAR_FULL*0.55, height=0.55, fill_color=BLUE, fill_opacity=0.8)

        raw_bar.next_to(comparison_title, DOWN, buff=0.5).to_edge(LEFT, buff=0.8)
        png_bar.next_to(raw_bar, DOWN, buff=0.3).to_edge(LEFT, buff=0.8)
        raw_bar.align_to(png_bar, LEFT)

        raw_label = Text("Dữ liệu thô (RGB thuần):  ~15 MB", font_size=20, color=RED)
        png_label_bar = Text("Sau nén PNG (~55%):  ~8.3 MB", font_size=20, color=BLUE)
        raw_label.next_to(raw_bar, RIGHT, buff=0.2)
        png_label_bar.next_to(png_bar, RIGHT, buff=0.2)

        with self.voiceover("Kết quả cuối cùng: Thuật toán PNG có thể co ngót kích\
 thước tệp từ 25 đến 50 phần trăm mà không đánh rơi dù chỉ một hạt nhiễu nhỏ nhất.") as tr:
            self.update_subtitle("PNG co ngót kích thước 25–50% mà không mất\ndù chỉ một bit dữ liệu!")
            self.play(Write(comparison_title))
            self.play(Create(raw_bar), Write(raw_label))
            self.play(Create(png_bar), Write(png_label_bar))

        # ── Minh họa trong suốt (Alpha channel) ─────────────────────────────
        checkerboard_group = VGroup()
        TILE = 0.35
        COLS_CB, ROWS_CB = 14, 8
        for row in range(ROWS_CB):
            for col in range(COLS_CB):
                color_cb = GRAY_D if (row + col) % 2 == 0 else WHITE
                t = Square(side_length=TILE)
                t.set_fill(color_cb, opacity=1)
                t.set_stroke(width=0)
                t.move_to(
                    RIGHT*(col - COLS_CB/2 + 0.5)*TILE +
                    UP*(row  - ROWS_CB/2 + 0.5)*TILE +
                    DOWN*1.3
                )
                checkerboard_group.add(t)

        # Đặt 1 ảnh thu nhỏ lên trên checkerboard
        try:
            thumb = ImageMobject(pil_img).scale_to_fit_height(2.4)
            thumb.move_to(DOWN*1.3)
        except Exception:
            thumb = Rectangle(width=3, height=2.4, fill_color=GRAY,
                              fill_opacity=0.5).move_to(DOWN*1.3)

        alpha_label = Text("Alpha channel → Trong suốt!", font_size=22, color=BLUE)
        alpha_label.next_to(checkerboard_group, DOWN, buff=0.2)

        transparent_note = Text("Cạnh sắc nét 100% – không viền mờ!",
                                 font_size=20, color=GREEN_C)
        transparent_note.next_to(alpha_label, DOWN, buff=0.15)

        with self.voiceover("PNG còn hỗ trợ kênh Alpha, cho phép tách nền trong suốt\
 với độ sắc nét tuyệt đối, không viền mờ. Đây là lý do PNG không thể thiếu trong\
 đồ họa thiết kế.") as tr:
            self.update_subtitle("PNG hỗ trợ kênh Alpha – trong suốt với độ sắc nét\ntuyệt đối. Không thể thiếu trong đồ họa!")
            self.play(
                FadeOut(raw_bar, png_bar, raw_label, png_label_bar)
            )
            self.play(FadeIn(checkerboard_group), run_time=1)
            self.play(FadeIn(thumb))
            self.play(Write(alpha_label), Write(transparent_note))

        # ── Tổng kết ─────────────────────────────────────────────────────────
        self.play(
            *[FadeOut(m) for m in [
                checkerboard_group, thumb, alpha_label, transparent_note,
                comparison_title
            ]]
        )

        summary = VGroup(
            Text("JPG  :  Nén tổn hao, tỉ lệ nén cao, mất data",
                 font_size=22, color=ORANGE),
            Text("PNG  :  Nén không tổn hao, lossless + Alpha",
                 font_size=22, color=BLUE),
            Text("Pixel = ma trận số  |  RGB Tensor  |  DCT  |  DEFLATE",
                 font_size=18, color=GRAY_A),
        ).arrange(DOWN, buff=0.3).move_to(ORIGIN)

        with self.voiceover("Tóm lại: JPG và PNG là hai trường phái hoàn toàn khác\
 nhau phục vụ những mục tiêu khác nhau. Hiểu bản chất toán học phía sau giúp bạn\
 lựa chọn định dạng phù hợp cho từng bài toán.") as tr:
            self.update_subtitle("JPG và PNG phục vụ mục tiêu khác nhau.\nHiểu toán học phía sau giúp bạn chọn đúng định dạng!")
            self.play(Write(summary[0]))
            self.play(Write(summary[1]))
            self.play(Write(summary[2]))

        self.wait(1.5)
        self.play(*[FadeOut(m) for m in self.mobjects])


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SCENE GỘP TOÀN BỘ PHẦN 2-3-4
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class FullVideoParts234(FaMIBaseScene):
    """
    Ghép Phần 2, 3, 4 liên tiếp trong một Scene.
    Dùng để render liền mạch cùng với BirthOfPixelScript (Phần 1).
    """
    def construct(self):
        RGBTensorScene.construct(self)
        JPGScene.construct(self)
        PNGScene.construct(self)

IndentationError: unexpected indent (<string>, line 1)